# AFIR for a molecule on a surface with NVIDIA ALCHEMI, on a platform GPU

Places a molecule at random over a slab many times and relaxes every placement with MACE through the
[NVIDIA ALCHEMI toolkit](https://github.com/NVIDIA/nvalchemi-toolkit), all placements batched on one GPU: first
under the artificial force (AFIR) between molecule and slab, then without it. The final structures are ranked by
energy.

The workflow has two steps, each in its own Python environment: *Configurations* builds the placements from the
job's two materials with made, and *AFIR (ALCHEMI)* relaxes them. This notebook builds the slab, fetches the
molecule, saves both as materials, creates the workflow, runs one job on the two materials and shows the ranked
configurations.

Outside the platform's JupyterLite, use Python 3.10 or 3.11 and install the packages once with `pip install "mat3ra-notebooks-utils[all]"`.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to use an organization as the owner, otherwise the personal account

# 3. Material parameters
CRYSTAL_NAME = "Cu"  # from Materials Standata
MILLER_INDICES = (1, 1, 1)
NUMBER_OF_LAYERS = 2
XY_SUPERCELL_MATRIX = [[4, 0], [0, 4]]
USE_CONVENTIONAL_CELL = False  # the supercell repeats the primitive surface cell
VACUUM = 15.0  # in angstroms
MOLECULE_NAME = "water"  # from PubChem

# 4. Workflow parameters
N_CONFIGURATIONS = 100  # random placements of the molecule over the slab
GAMMA = 100  # kJ/mol, model collision energy of the AFIR force
SEED = 0  # of the random placements
PYTHON_VERSION = "3.11.10"
PYTHON_BUILD = "Default"

# 5. Compute parameters
CLUSTER_NAME = "cluster-003"
QUEUE_NAME = QueueName.GOF  # H100 nodes on cluster-003 (GOF on-demand, GSF spot)
TIME_LIMIT = "01:00:00"

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Build the slab

In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.helpers import create_slab
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.standata.materials import Materials

crystal = Material.create(Materials.get_by_name_first_match(CRYSTAL_NAME))
slab = create_slab(
    crystal,
    miller_indices=MILLER_INDICES,
    number_of_layers=NUMBER_OF_LAYERS,
    vacuum=VACUUM,
    xy_supercell_matrix=XY_SUPERCELL_MATRIX,
    use_conventional_cell=USE_CONVENTIONAL_CELL,
)
visualize(slab)

### 3.2. Fetch the molecule from PubChem

In [ ]:
import io
from urllib.parse import quote

from ase.io import read
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.io import read_from_url

PUBCHEM_STRUCTURE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF?record_type=3d"

molecule_sdf = await read_from_url(PUBCHEM_STRUCTURE_URL.format(name=quote(MOLECULE_NAME)))
molecule = Material.create(from_ase(read(io.StringIO(molecule_sdf), format="sdf")))
molecule.name = MOLECULE_NAME
visualize(molecule)

### 3.3. Save materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in (slab, molecule)]

## 4. Create the workflow
### 4.1. Step 1: configurations

Places the molecule `N_CONFIGURATIONS` times over the slab, each at a random orientation and a random point of the
surface cell, with its lowest atom above the top slab atom by the sum of their covalent radii plus 0.5 Å. Slab atoms
are labelled 0, molecule atoms 1.

In [ ]:
CONFIGURATIONS_SCRIPT = r'''"""Places the job's molecule over its slab N_CONFIGURATIONS times, each at a random orientation and a random point of
the surface cell, and passes the configurations to the next step as CONFIGURATIONS."""

import json

import numpy as np
from ase.data import atomic_numbers, covalent_radii
from mat3ra.made.material import Material
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.made.tools.modify import translate_by_vector
from mat3ra.made.tools.operations.core.binary import merge
from mat3ra.made.tools.operations.core.unary import rotate

N_CONFIGURATIONS = 100
SEED = 0
CONTACT_OFFSET = 0.5  # A, lowest molecule atom at R_molecule + R_slab + CONTACT_OFFSET above the top slab atom (GRRM17 RandDC)
# The job's materials, written here by the platform when the job is created.
MATERIALS = json.loads(r"""{{ materials | dump | safe }}""")


def material_in_crystal_units(document):
    material = Material.create(document)
    material.to_crystal()
    return material


def extreme_atom_height_and_radius(material, choose):
    cartesian = material.clone()
    cartesian.to_cartesian()
    height, element = choose(zip([coordinate[2] for coordinate in cartesian.basis.coordinates.values],
                                 cartesian.basis.elements.values))
    return height, covalent_radii[atomic_numbers[element]]


def create_configuration(slab, molecule, random_generator):
    quaternion = random_generator.normal(size=4)
    angle = 2 * np.degrees(np.arccos(quaternion[0] / np.linalg.norm(quaternion)))
    rotated = rotate(molecule, axis=quaternion[1:].tolist(), angle=angle, wrap=False)
    rotated.set_lattice(slab.lattice)
    top_height, top_radius = extreme_atom_height_and_radius(slab, max)
    lowest_height, lowest_radius = extreme_atom_height_and_radius(rotated, min)
    point = slab.basis.cell.convert_point_to_cartesian([random_generator.random(), random_generator.random(), 0.0])
    center = molecule.basis.cell.convert_point_to_cartesian([0.5, 0.5, 0.5])
    height = top_height + top_radius + lowest_radius + CONTACT_OFFSET - lowest_height
    placed = translate_by_vector(rotated, vector=[point[0] - center[0], point[1] - center[1], height],
                                 use_cartesian_coordinates=True)
    placed.set_labels_from_value(InterfacePartsEnum.FILM.value)
    return merge([slab, placed])


def main():
    slab = material_in_crystal_units(next(item for item in MATERIALS if not item.get("isNonPeriodic")))
    molecule = material_in_crystal_units(next(item for item in MATERIALS if item.get("isNonPeriodic")))
    slab.set_labels_from_value(InterfacePartsEnum.SUBSTRATE.value)
    random_generator = np.random.default_rng(SEED)
    configurations = [create_configuration(slab, molecule, random_generator).to_dict() for _ in range(N_CONFIGURATIONS)]
    print(json.dumps({"name": "CONFIGURATIONS", "value": configurations, "scope": "global"}))


if __name__ == "__main__":
    main()
'''

In [ ]:
CONFIGURATIONS_REQUIREMENTS = "mat3ra-made[tools]\n"

### 4.2. Step 2: AFIR with ALCHEMI

Relaxes all configurations together with the slab fixed, first under the AFIR force of strength `GAMMA` between
molecule and slab, then without it, and writes `results.csv` and `final_structures.extxyz`. Another backend or
another ALCHEMI version replaces this script and its packages; step 1 stays.

In [ ]:
AFIR_SCRIPT = r'''"""Relaxes the configurations of the previous step with MACE, all batched with the slab fixed: first under the AFIR
force between molecule and slab, then without it. Ranks the final structures by energy."""

import csv
import json

import numpy as np
import torch
from ase.calculators.singlepoint import SinglePointCalculator
from ase.data import covalent_radii
from ase.io import write
from loguru import logger
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from nvalchemi._typing import AtomCategory
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import FIRE2, FusedStage
from nvalchemi.dynamics.base import ConvergenceHook, DynamicsStage
from nvalchemi.dynamics.hooks import FreezeAtomsHook
from nvalchemi.hooks import BiasedPotentialHook
from nvalchemi.models.mace import MACEWrapper

DEVICE = "cuda:0"
CHECKPOINT = "medium-mpa-0"
DTYPE = torch.float32
FMAX = 0.03  # eV/A
MAX_STEPS = 500
TIME_STEP = 0.5
GAMMA = 100  # kJ/mol, model collision energy of the AFIR force
AFIR_R0 = 3.8164  # A, GRRM17 Eq. (3)
AFIR_EPSILON = 1.0061  # kJ/mol, GRRM17 Eq. (3)
KJ_PER_MOL_PER_EV = 96.485
CONFIGURATIONS = json.loads(r"""{{ CONFIGURATIONS | tojson }}""")

ALPHA = GAMMA / ((2 ** (-1 / 6) - (1 + (1 + GAMMA / AFIR_EPSILON) ** 0.5) ** (-1 / 6)) * AFIR_R0) / KJ_PER_MOL_PER_EV
CATEGORY_BY_PART = {InterfacePartsEnum.SUBSTRATE.value: AtomCategory.SURFACE.value,
                    InterfacePartsEnum.FILM.value: AtomCategory.GAS.value}
IN_PLANE = torch.tensor([1.0, 1.0, 0.0], device=DEVICE)
IN_PLANE_SHIFTS = torch.tensor([[a, b, 0.0] for a in (-1, 0, 1) for b in (-1, 0, 1)], device=DEVICE)
COVALENT_RADII = torch.tensor(covalent_radii, device=DEVICE, dtype=DTYPE)


def in_plane_minimum_image(vectors, cells):
    fractional = torch.einsum("pi,pij->pj", vectors, torch.linalg.inv(cells))
    fractional = fractional - torch.round(fractional) * IN_PLANE
    images = torch.einsum("psi,pij->psj", fractional.unsqueeze(1) + IN_PLANE_SHIFTS, cells)
    nearest = images.norm(dim=-1).argmin(dim=1)
    return torch.take_along_dim(images, nearest[:, None, None], dim=1).squeeze(1)


def afir_energy(positions, batch):
    molecule_atoms = torch.where(batch.atom_categories == AtomCategory.GAS.value)[0]
    surface_atoms = torch.where(batch.atom_categories == AtomCategory.SURFACE.value)[0]
    molecule_index, surface_index = torch.where(
        batch.batch_idx[molecule_atoms][:, None] == batch.batch_idx[surface_atoms][None, :])
    pair_molecule_atoms, pair_surface_atoms = molecule_atoms[molecule_index], surface_atoms[surface_index]
    pair_structures = batch.batch_idx[pair_molecule_atoms]
    distances = in_plane_minimum_image(positions[pair_molecule_atoms] - positions[pair_surface_atoms],
                                       batch.cell[pair_structures]).norm(dim=-1)
    radii = (COVALENT_RADII[batch.atomic_numbers[pair_molecule_atoms]]
             + COVALENT_RADII[batch.atomic_numbers[pair_surface_atoms]])
    weights = (radii / distances) ** 6
    numerator = torch.zeros(batch.num_graphs, device=positions.device, dtype=positions.dtype)
    denominator = torch.zeros_like(numerator)
    return ALPHA * (numerator.index_add(0, pair_structures, weights * distances)
                    / denominator.index_add(0, pair_structures, weights))


def afir_bias(batch):
    positions = batch.positions.clone().requires_grad_(True)
    energy = afir_energy(positions, batch)
    forces = -torch.autograd.grad(energy.sum(), positions)[0]
    return energy.detach().unsqueeze(-1), forces


def atomic_data(atoms):
    data = AtomicData.from_atoms(atoms, device=DEVICE, dtype=DTYPE)
    data["atom_categories"] = torch.tensor([CATEGORY_BY_PART[tag] for tag in atoms.get_tags()], device=DEVICE)
    data["forces"] = torch.zeros_like(data.positions)
    data["energy"] = torch.zeros((1, 1), device=DEVICE, dtype=DTYPE)
    return data


def relaxed_atoms(original, row):
    atoms = original.copy()
    atoms.positions = row.positions.cpu().numpy()
    atoms.calc = SinglePointCalculator(atoms, energy=float(row.energy.item()), forces=row.forces.cpu().numpy())
    return atoms


def relax(structures, model, bias_hooks):
    optimizer = FIRE2(model=model, dt=TIME_STEP, n_steps=MAX_STEPS, convergence_hook=ConvergenceHook(
        criteria=[{"key": "forces", "threshold": FMAX, "reduce_op": "norm", "reduce_dims": -1}]))
    optimizer.register_hook(FreezeAtomsHook(freeze_category=AtomCategory.SURFACE.value))
    stage = FusedStage(sub_stages=[(0, optimizer)])
    for hook in model.make_neighbor_hooks():
        stage.register_hook(hook, stage=DynamicsStage.BEFORE_COMPUTE)
    for hook in bias_hooks:
        stage.register_hook(hook, stage=DynamicsStage.AFTER_COMPUTE)
    batch = stage.run(Batch.from_data_list([atomic_data(atoms) for atoms in structures]))
    return [relaxed_atoms(atoms, row) for atoms, row in zip(structures, batch.to_data_list())]


def energy_and_max_force(atoms):
    return atoms.get_potential_energy(), np.linalg.norm(atoms.get_forces(), axis=1).max()


def write_results(afir_structures, final_structures):
    order = np.argsort([atoms.get_potential_energy() for atoms in final_structures])
    with open("results.csv", "w", newline="") as results_file:
        writer = csv.writer(results_file)
        writer.writerow(["configuration", "rank", "afir_function_eV", "afir_max_force_eV_per_A", "energy_eV",
                         "max_force_eV_per_A"])
        for rank, index in enumerate(order, start=1):
            writer.writerow([index, rank, *energy_and_max_force(afir_structures[index]),
                             *energy_and_max_force(final_structures[index])])
            final_structures[index].info.update(configuration=int(index), rank=rank)
    write("final_structures.extxyz", [final_structures[index] for index in order])


def main():
    logger.info("AFIR force α, γ = {} kJ/mol: {:.3f} eV/Å", GAMMA, ALPHA)
    structures = [to_ase(Material.create(configuration)) for configuration in CONFIGURATIONS]
    model = MACEWrapper.from_checkpoint(checkpoint_path=CHECKPOINT, device=torch.device(DEVICE), dtype=DTYPE)
    afir_structures = relax(structures, model, [BiasedPotentialHook(bias_fn=afir_bias)])
    final_structures = relax(afir_structures, model, [])
    write_results(afir_structures, final_structures)


if __name__ == "__main__":
    main()
'''

In [ ]:
AFIR_REQUIREMENTS = """--find-links https://download.pytorch.org/whl/cu128/torch/
--find-links https://download.pytorch.org/whl/cu128/torchvision/
--find-links https://download.pytorch.org/whl/cu128/torchaudio/
--find-links https://github.com/NVIDIA/warp/releases/expanded_assets/v1.18.0
torch==2.11.0+cu128
torchvision==0.26.0+cu128
torchaudio==2.11.0+cu128
warp-lang==1.18.0+cu12
nvalchemi-toolkit[mace,ase]==0.2.0
matplotlib
numpy
ase
mace-torch
loguru
mat3ra-made[tools]
"""

### 4.3. Create the workflow

When the job is created, the platform writes the job's materials into step 1's script (`materials`; the
non-periodic one is the molecule). Step 1 prints `CONFIGURATIONS` to the job's scope, and the compute node writes
them into step 2's script before it runs.

In [ ]:
import copy
import re

from mat3ra.notebooks_utils.core.entity.workflow.api import set_execution_unit_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.standata.subworkflows import SubworkflowStandata
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode import Subworkflow, Workflow

workflow_config = copy.deepcopy(WorkflowStandata.filter_by_application("python").get_by_name_first_match("custom_script.json"))
workflow_config.update(name="AFIR, molecule on surface", isMultiMaterial=True)
workflow_config["units"][0]["name"] = "Configurations"
configurations_subworkflow = workflow_config["subworkflows"][0]
afir_subworkflow = copy.deepcopy(SubworkflowStandata.filter_by_application("python").get_by_name_first_match("python_script.json"))
afir_results = [
    {"name": "file_content", "filetype": "csv", "basename": "results.csv"},
    {"name": "file_content", "filetype": "text", "basename": "final_structures.extxyz"},
]
for subworkflow, name, unit_name, requirements, results in (
    (configurations_subworkflow, "Configurations", "configurations", CONFIGURATIONS_REQUIREMENTS, []),
    (afir_subworkflow, "AFIR (ALCHEMI)", "afir", AFIR_REQUIREMENTS, afir_results),
):
    unit = subworkflow["units"][-1]
    unit.update(name=unit_name, head=True, results=results)
    for application in (subworkflow["application"], unit["application"]):
        application.update(version=PYTHON_VERSION, build=PYTHON_BUILD)
    set_execution_unit_input(unit, "requirements.txt", requirements)
    subworkflow.update(name=name, units=[unit])

configurations_script = re.sub(r"^N_CONFIGURATIONS = \S+", f"N_CONFIGURATIONS = {N_CONFIGURATIONS}", CONFIGURATIONS_SCRIPT, flags=re.M)
script_input = next(unit_input for unit_input in configurations_subworkflow["units"][0]["input"] if unit_input["template"]["name"] == "script.py")
script_input["template"]["content"] = re.sub(r"^SEED = \S+", f"SEED = {SEED}", configurations_script, flags=re.M)
set_execution_unit_input(afir_subworkflow["units"][0], "script.py", re.sub(r"^GAMMA = \S+", f"GAMMA = {GAMMA}", AFIR_SCRIPT, flags=re.M))

workflow = Workflow.create(workflow_config)
workflow.add_subworkflow(Subworkflow(**afir_subworkflow))
visualize_workflow(workflow)

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")

### 5.2. Create compute configuration for the job

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next(cluster for cluster in clusters if CLUSTER_NAME in cluster["hostname"])
compute = Compute(cluster=cluster, queue=QUEUE_NAME, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {compute.nodes}, ppn: {compute.ppn}, time limit: {TIME_LIMIT}")

## 6. Create the job

In [ ]:
from mat3ra.notebooks_utils.job import create_job

job = create_job(
    api_client=client,
    materials=saved_materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=f"{workflow.name} {MOLECULE_NAME} {N_CONFIGURATIONS} {timestamp}",
    compute=compute.to_dict(),
)
job_id = job["_id"]
print(f"✅ Created job: {job_id}")

## 7. Submit the job and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, [job_id])

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Results
### 8.1. Configurations ranked by final energy

`afir_function_eV` is the MACE energy plus the AFIR term at the end of the first relaxation, `energy_eV` the MACE
energy of the final structure; the `max_force` columns give the largest force on an atom at the end of each.

In [ ]:
import pandas as pd

files = {file["key"].rsplit("/", 1)[-1]: file["signedUrl"] for file in client.jobs.list_files(job_id)}
pd.read_csv(io.StringIO(await read_from_url(files["results.csv"])))

## References

[1] S. Maeda, Y. Harabuchi, M. Takagi, K. Saita, K. Suzuki, T. Ichino, Y. Sumiya, K. Sugiyama and Y. Ono,
"Implementation and performance of the artificial force induced reaction method in the GRRM17 program",
J. Comput. Chem. 39, 233 (2018). https://doi.org/10.1002/jcc.25106

[2] S. Maeda and K. Morokuma, "Communications: A systematic method for locating transition structures of A + B → X
type reactions", J. Chem. Phys. 132, 241102 (2010). https://doi.org/10.1063/1.3457903

[3] K. Sugiyama, Y. Sumiya, M. Takagi, K. Saita and S. Maeda, "Understanding CO oxidation on the Pt(111) surface
based on a reaction route network", Phys. Chem. Chem. Phys. 21, 14366 (2019). https://doi.org/10.1039/c8cp06856a